In [5]:
!pip install langchain chromadb openai tiktoken pypdf langchain_huggingface langchain-community langchain_core langchain-chroma

In [6]:
import os
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# Load environment variables
load_dotenv()

False

In [7]:
doc1 = Document(
    page_content="""
    Virat Kohli is one of the most successful and consistent batsmen in IPL history.
    Known for his aggressive batting style and fitness.
    """,
    metadata={"team": "Royal Challengers Bangalore"}
)

doc2 = Document(
    page_content="""
    MS Dhoni is one of the greatest captains in cricket history.
    Known for calm leadership and finishing ability.
    """,
    metadata={"team": "Chennai Super Kings"}
)

doc3 = Document(
    page_content="""
    Jasprit Bumrah is regarded as one of the best fast bowlers in modern cricket.
    Famous for yorkers and death bowling.
    """,
    metadata={"team": "Mumbai Indians"}
)
docs = [doc1, doc2, doc3]


In [4]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [8]:
vector_store = Chroma(
    collection_name="ipl_players",
    embedding_function=embeddings,
    persist_directory="my_chroma_db"
)

In [9]:
vector_store.add_documents(docs)


['93fbc207-43e8-4493-8d0b-2c56bb50acac',
 '2557cd17-2dde-4569-ac3a-198aff90f776',
 '179bafe7-a16c-4502-8b2d-5c8026196081']

In [10]:
results = vector_store.similarity_search(
    query="Who is a fast bowler?",
    k=2
)
results

[Document(id='179bafe7-a16c-4502-8b2d-5c8026196081', metadata={'team': 'Mumbai Indians'}, page_content='\n    Jasprit Bumrah is regarded as one of the best fast bowlers in modern cricket.\n    Famous for yorkers and death bowling.\n    '),
 Document(id='93fbc207-43e8-4493-8d0b-2c56bb50acac', metadata={'team': 'Royal Challengers Bangalore'}, page_content='\n    Virat Kohli is one of the most successful and consistent batsmen in IPL history.\n    Known for his aggressive batting style and fitness.\n    ')]

In [11]:
for doc in results:
    print(doc.page_content)
    print(doc.metadata)
    print("-" * 50)


    Jasprit Bumrah is regarded as one of the best fast bowlers in modern cricket.
    Famous for yorkers and death bowling.
    
{'team': 'Mumbai Indians'}
--------------------------------------------------

    Virat Kohli is one of the most successful and consistent batsmen in IPL history.
    Known for his aggressive batting style and fitness.
    
{'team': 'Royal Challengers Bangalore'}
--------------------------------------------------


In [12]:
# Similarity Search with Score

results_with_score = vector_store.similarity_search_with_score(
    query="Who is a captain?",
    k=2
)
results_with_score

[(Document(id='2557cd17-2dde-4569-ac3a-198aff90f776', metadata={'team': 'Chennai Super Kings'}, page_content='\n    MS Dhoni is one of the greatest captains in cricket history.\n    Known for calm leadership and finishing ability.\n    '),
  0.9674180746078491),
 (Document(id='93fbc207-43e8-4493-8d0b-2c56bb50acac', metadata={'team': 'Royal Challengers Bangalore'}, page_content='\n    Virat Kohli is one of the most successful and consistent batsmen in IPL history.\n    Known for his aggressive batting style and fitness.\n    '),
  1.4562506675720215)]

In [13]:
for doc, score in results_with_score:
    print("Score:", score)
    print(doc.page_content)
    print(doc.metadata)
    print("-" * 50)

Score: 0.9674180746078491

    MS Dhoni is one of the greatest captains in cricket history.
    Known for calm leadership and finishing ability.
    
{'team': 'Chennai Super Kings'}
--------------------------------------------------
Score: 1.4562506675720215

    Virat Kohli is one of the most successful and consistent batsmen in IPL history.
    Known for his aggressive batting style and fitness.
    
{'team': 'Royal Challengers Bangalore'}
--------------------------------------------------


In [14]:
# Metadata Filtering

filtered_docs = vector_store.similarity_search(
    query="",
    filter={"team": "Chennai Super Kings"}
)
filtered_docs

[Document(id='2557cd17-2dde-4569-ac3a-198aff90f776', metadata={'team': 'Chennai Super Kings'}, page_content='\n    MS Dhoni is one of the greatest captains in cricket history.\n    Known for calm leadership and finishing ability.\n    ')]

In [15]:
for doc in filtered_docs:
    print(doc.page_content)


    MS Dhoni is one of the greatest captains in cricket history.
    Known for calm leadership and finishing ability.
    
